**IMPORTS AND LOAD**

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/cleaned_data.csv",
    dtype={"diag_1": str, "diag_2": str, "diag_3": str},
)
print(df.shape)
df.head()

(69987, 47)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,...,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,target
0,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,41,0,...,No,No,No,No,No,No,No,No,NO,0
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,59,0,...,Up,No,No,No,No,No,Ch,Yes,>30,0
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,11,5,...,No,No,No,No,No,No,No,Yes,NO,0
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,44,1,...,Up,No,No,No,No,No,Ch,Yes,NO,0
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,51,0,...,Steady,No,No,No,No,No,Ch,Yes,NO,0


**AGE BRACKET TO A NUMBER**

In [2]:
# "[70-80)" -> 75
df["age_num"] = df["age"].str.extract(r"\[(\d+)-")[0].astype(int) + 5
df[["age", "age_num"]].drop_duplicates().sort_values("age_num")

,age,age_num
0,[0-10),5
1,[10-20),15
2,[20-30),25
3,[30-40),35
4,[40-50),45
5,[50-60),55
6,[60-70),65
7,[70-80),75
8,[80-90),85
9,[90-100),95


**PRIOR VISITS**

In [3]:
df["prior_visits"] = (
    df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
)
df["had_prior_inpatient"] = (df["number_inpatient"] > 0).astype(int)

**GROUP DIAGNOSIS CODES INTO CATEGORIES**

In [4]:
def icd9_group(code):
    if code == "Unknown":
        return "Unknown"
    if code.startswith(("V", "E")):
        return "Other"
    try:
        c = float(code)
    except ValueError:
        return "Other"
    if 390 <= c < 460 or int(c) == 785: return "Circulatory"
    if 460 <= c < 520 or int(c) == 786: return "Respiratory"
    if 520 <= c < 580 or int(c) == 787: return "Digestive"
    if int(c) == 250:                   return "Diabetes"
    if 800 <= c < 1000:                 return "Injury"
    if 710 <= c < 740:                  return "Musculoskeletal"
    if 580 <= c < 630 or int(c) == 788: return "Genitourinary"
    if 140 <= c < 240:                  return "Neoplasms"
    return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    df[col + "_group"] = df[col].apply(icd9_group)

df["diag_1_group"].value_counts()

diag_1_group
Circulatory        21389
Other              12124
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Unknown               10
Name: count, dtype: int64

**MEDICATION FEATURES**

In [5]:
med_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "examide", "citoglipton", "insulin", "glyburide-metformin",
    "glipizide-metformin", "glimepiride-pioglitazone",
    "metformin-rosiglitazone", "metformin-pioglitazone",
]

df["n_meds_taken"] = (df[med_cols] != "No").sum(axis=1)
df["n_meds_changed"] = df[med_cols].isin(["Up", "Down"]).sum(axis=1)

# these two columns are "No" for every patient, so they carry no information
print(df[["examide", "citoglipton"]].nunique())
df = df.drop(columns=med_cols)

examide        1
citoglipton    1
dtype: int64


**LAB TEST FLAGS**

In [6]:
df["a1c_tested"] = (df["A1Cresult"] != "None").astype(int)
df["glucose_tested"] = (df["max_glu_serum"] != "None").astype(int)

**GROUP RARE CATEGORIES INTO "OTHER"**

In [7]:
def group_rare(series, top_n):
    keep = series.value_counts().nlargest(top_n).index
    return series.where(series.isin(keep), "Other")

df["medical_specialty"] = group_rare(df["medical_specialty"], 10)
df["discharge_disposition_id"] = group_rare(df["discharge_disposition_id"].astype(str), 6)
df["admission_source_id"] = group_rare(df["admission_source_id"].astype(str), 5)
df["admission_type_id"] = df["admission_type_id"].astype(str)

**REMOVE UNNECESSARY COLUMNS**

In [8]:
df = df.drop(columns=["readmitted", "age", "diag_1", "diag_2", "diag_3"])
df.columns.tolist()

['race',
 'gender',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'change',
 'diabetesMed',
 'target',
 'age_num',
 'prior_visits',
 'had_prior_inpatient',
 'diag_1_group',
 'diag_2_group',
 'diag_3_group',
 'n_meds_taken',
 'n_meds_changed',
 'a1c_tested',
 'glucose_tested']

**LIST THE COLUMN TYPES**

In [9]:
target = "target"
cat_cols = df.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in df.columns if c not in cat_cols + [target]]

print("Categorical:", cat_cols)
print("Numeric:", num_cols)
print("Total features:", len(cat_cols) + len(num_cols))
print("Missing values:", df.isnull().sum().sum())

Categorical: ['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed', 'diag_1_group', 'diag_2_group', 'diag_3_group']
Numeric: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_num', 'prior_visits', 'had_prior_inpatient', 'n_meds_taken', 'n_meds_changed', 'a1c_tested', 'glucose_tested']
Total features: 28
Missing values: 123779


C:\Users\Badaplin\AppData\Local\Temp\ipykernel_32456\47138144.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns.tolist()


**SANITY CHECK THAT THE NEW FEATURES CARRY SIGNAL**

In [10]:
print(df.groupby("prior_visits")["target"].mean().head(8).round(3))
print(df.groupby("diag_1_group")["target"].mean().round(3).sort_values(ascending=False))

prior_visits
0    0.079
1    0.106
2    0.122
3    0.128
4    0.146
5    0.138
6    0.191
7    0.196
Name: target, dtype: float64
diag_1_group
Unknown            0.200
Injury             0.108
Circulatory        0.097
Diabetes           0.091
Neoplasms          0.091
Genitourinary      0.090
Other              0.090
Musculoskeletal    0.084
Digestive          0.080
Respiratory        0.073
Name: target, dtype: float64


**SAVE THE RESULTS**

In [11]:
df.to_csv("../data/features.csv", index=False)
print(df.shape)

(69987, 29)
